In [192]:
from pathlib import Path
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from collections import defaultdict
import pybedtools

In [221]:
ROOT = Path().cwd()
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
RNACHROM_INITIAL = DATA_INITIAL / "rnachrom"
PREPROCESSING = Path("../preprocessing")
RNACHROM_INDEX_BED = PREPROCESSING / "rnachrom_index-bed"
RNACHROM_INDEX_BED_V2 = PREPROCESSING / "rnachrom_index-bed-v2"
GENCODE = DATA / "gencode"

In [223]:
for directory in DATA_INITIAL, DATA, RNACHROM_INITIAL, PREPROCESSING, RNACHROM_INDEX_BED, RNACHROM_INDEX_BED_V2:
    directory.mkdir(exist_ok=True, parents=True)

In [111]:
!ls {RNACHROM_INITIAL}/experiment_all_to_all_id_10_method_Red_C_organism_Homo_sapiens_tissue_K562/contacts

K562.chr10.tab	K562.chr15.tab	K562.chr1.tab	K562.chr3.tab  K562.chr8.tab
K562.chr11.tab	K562.chr16.tab	K562.chr20.tab	K562.chr4.tab  K562.chr9.tab
K562.chr12.tab	K562.chr17.tab	K562.chr21.tab	K562.chr5.tab  K562.chrX.tab
K562.chr13.tab	K562.chr18.tab	K562.chr22.tab	K562.chr6.tab  K562.chrY.tab
K562.chr14.tab	K562.chr19.tab	K562.chr2.tab	K562.chr7.tab


In [7]:
!head {RNACHROM_INITIAL}/experiment_all_to_all_id_10_method_Red_C_organism_Homo_sapiens_tissue_K562/contacts/K562.chrY.tab

rna_chr	rna_start	rna_end	dna_chr	dna_start	dna_end	sample	gene_name	gene_type	gene_strand	source	expected_value	norm_value
chrY	11177322	11177366	chr2	61571545	61571568	GSM4041591	AC134878.2	transcribed_unprocessed_pseudogene	-	gencode	0.19	1.09
chrY	11169441	11169485	chr15	51880717	51880740	GSM4041591	AC134878.2	transcribed_unprocessed_pseudogene	-	gencode	0.308	0.93
chrY	11190749	11190793	chr18	73987211	73987235	GSM4041591	AC134878.2	transcribed_unprocessed_pseudogene	-	gencode	0.119	1.215
chrY	11208920	11208965	chr12	124506997	124507020	GSM4041591	AC134878.2	transcribed_unprocessed_pseudogene	-	gencode	0.213	1.054
chrY	11198155	11198198	chr6	31611333	31611357	GSM4041591	AC134878.2	transcribed_unprocessed_pseudogene	-	gencode	0.307	0.932
chrY	11204140	11204185	chr18	57700628	57700651	GSM4041591	AC134878.2	transcribed_unprocessed_pseudogene	-	gencode	0.255	0.996
chrY	11162679	11162725	chr2	190488939	190488961	GSM4041591	AC134878.2	transcribed_unprocessed_pseudogene	-	gencode	0.345	0.

gene_name - это имя гена РНК!!!

In [16]:
to_file = GENCODE / 'gencode_v35-to-v47-genes-hg38-with_names-all_info.tsv'
gencode_v35_to_v47_genes_hg38_with_names = pd.read_csv(to_file, sep="\t", index_col=False)
gencode_v35_to_v47_genes_hg38_with_names.head(3)

,chrom,start,end,score,strand,ens_gene,gene_name,new-ens_gene,new-gene_name
0,chr1,11869,14409,.,+,ENSG00000223972,DDX11L1,ENSG00000223972,DDX11L1
1,chr1,14404,29570,.,-,ENSG00000227232,WASH7P,ENSG00000227232,WASH7P
2,chr1,17369,17436,.,-,ENSG00000278267,MIR6859-1,ENSG00000278267,MIR6859-1


In [17]:
len(gencode_v35_to_v47_genes_hg38_with_names)

60656

In [252]:
gencode_v35_to_v47_genes_hg38_with_names[gencode_v35_to_v47_genes_hg38_with_names["new-ens_gene"] == "ENSG00000281849"]

,chrom,start,end,score,strand,ens_gene,gene_name,new-ens_gene,new-gene_name
57634,chrX,386980,405579,.,+,ENSG00000281849,AL732314.6,ENSG00000281849,ENSG00000281849
57636,chrX,419157,421980,.,+,ENSG00000280767,AL732314.4,ENSG00000281849,ENSG00000281849
60058,chrY,386980,405579,.,+,ENSG00000281849,AL732314.6,ENSG00000281849,ENSG00000281849
60060,chrY,419157,421980,.,+,ENSG00000280767,AL732314.4,ENSG00000281849,ENSG00000281849


In [63]:
to_himorna_rnachrom_mapping = RNACHROM_INITIAL / "lncRNA_CorrespondenceTable.tsv"
himorna_rnachrom_mapping = pd.read_csv(to_himorna_rnachrom_mapping, sep="\t")
himorna_rnachrom_mapping.head(2)

,chr_rna HimoRNA,start_rna HimoRNA,end_rna HimoRNA,str,gene_name HimoRNA,Ensembl ID HimoRNA,chr_rna RNA-Chrom,start_rna RNA-Chrom,end_rna RNA-Chrom,gene_name RNA-Chrom,Ensembl ID RNA-Chrom,Jaccard index
0,chr1,89295,133723,-,AL627309.1,ENSG00000238009.6,chr1,89295.0,133723.0,AL627309.1,ENSG00000238009.6,1.0
1,chr1,89551,91105,-,AL627309.3,ENSG00000239945.1,chr1,89551.0,91105.0,AL627309.3,ENSG00000239945.1,1.0


In [64]:
himorna_rnachrom_mapping.shape

(4149, 12)

In [65]:
himorna_rnachrom_mapping = himorna_rnachrom_mapping[["gene_name HimoRNA", "Ensembl ID HimoRNA", "gene_name RNA-Chrom", "Ensembl ID RNA-Chrom"]]

In [66]:
%%time
def drop_ensembl_version(x):
    return x.split(".")[0] if isinstance(x, str) else x

for col in ["Ensembl ID HimoRNA", "Ensembl ID RNA-Chrom"]:
    himorna_rnachrom_mapping[col] = himorna_rnachrom_mapping[col].apply(drop_ensembl_version)

CPU times: user 6.77 ms, sys: 2.01 ms, total: 8.78 ms
Wall time: 8.42 ms


In [67]:
himorna_rnachrom_mapping

,gene_name HimoRNA,Ensembl ID HimoRNA,gene_name RNA-Chrom,Ensembl ID RNA-Chrom
0,AL627309.1,ENSG00000238009,AL627309.1,ENSG00000238009
1,AL627309.3,ENSG00000239945,AL627309.3,ENSG00000239945
2,AL627309.2,ENSG00000239906,AL627309.2,ENSG00000239906
3,AL669831.5,ENSG00000237491,LINC01409,ENSG00000237491
4,FAM87B,ENSG00000177757,FAM87B,ENSG00000177757
...,...,...,...,...
4144,AL080317.1,ENSG00000230177,NaN,NaN
4145,AL356599.1,ENSG00000235652,NaN,NaN
4146,NKAIN3-IT1,ENSG00000274956,NaN,NaN
4147,AC010980.1,ENSG00000267034,NaN,NaN


In [68]:
himorna_rnachrom_mapping["Ensembl ID HimoRNA"].nunique()

4145

In [69]:
himorna_rnachrom_mapping["Ensembl ID RNA-Chrom"].nunique()

4120

In [70]:
himorna_rnachrom_mapping

,gene_name HimoRNA,Ensembl ID HimoRNA,gene_name RNA-Chrom,Ensembl ID RNA-Chrom
0,AL627309.1,ENSG00000238009,AL627309.1,ENSG00000238009
1,AL627309.3,ENSG00000239945,AL627309.3,ENSG00000239945
2,AL627309.2,ENSG00000239906,AL627309.2,ENSG00000239906
3,AL669831.5,ENSG00000237491,LINC01409,ENSG00000237491
4,FAM87B,ENSG00000177757,FAM87B,ENSG00000177757
...,...,...,...,...
4144,AL080317.1,ENSG00000230177,NaN,NaN
4145,AL356599.1,ENSG00000235652,NaN,NaN
4146,NKAIN3-IT1,ENSG00000274956,NaN,NaN
4147,AC010980.1,ENSG00000267034,NaN,NaN


In [71]:
himorna_rnachrom_mapping_updated = pd.merge(left=himorna_rnachrom_mapping, right=gencode_v35_to_v47_genes_hg38_with_names[["ens_gene","gene_name","new-ens_gene","new-gene_name"]], how="inner", left_on="Ensembl ID RNA-Chrom", right_on="ens_gene")
himorna_rnachrom_mapping_updated.head(1)

,gene_name HimoRNA,Ensembl ID HimoRNA,gene_name RNA-Chrom,Ensembl ID RNA-Chrom,ens_gene,gene_name,new-ens_gene,new-gene_name
0,AL627309.1,ENSG00000238009,AL627309.1,ENSG00000238009,ENSG00000238009,AL627309.1,ENSG00000241860,ENSG00000241860


In [72]:
len(himorna_rnachrom_mapping)

4149

In [78]:
mask = himorna_rnachrom_mapping_updated["Ensembl ID RNA-Chrom"] != himorna_rnachrom_mapping_updated["ens_gene"]
assert mask.sum() == 0

In [82]:
mask = himorna_rnachrom_mapping_updated["gene_name RNA-Chrom"] != himorna_rnachrom_mapping_updated["gene_name"]
mask.sum()

33

In [84]:
del himorna_rnachrom_mapping_updated["ens_gene"]

In [251]:
mask = himorna_rnachrom_mapping_updated["new-ens_gene"].apply(lambda x: "ENSG" in x)
himorna_rnachrom_mapping_updated[mask][himorna_rnachrom_mapping_updated[mask].duplicated()]

,gene_name HimoRNA,Ensembl ID HimoRNA,gene_name RNA-Chrom,Ensembl ID RNA-Chrom,gene_name,new-ens_gene,new-gene_name
4014,AL732314.6,ENSG00000281849,AL732314.6_1,ENSG00000281849,AL732314.6,ENSG00000281849,ENSG00000281849
4016,LINC00106,ENSG00000236871,LINC00106_1,ENSG00000236871,LINC00106,ENSG00000236871,LINC00106
4018,ASMTL-AS1,ENSG00000236017,ASMTL-AS1_1,ENSG00000236017,ASMTL-AS1,ENSG00000236017,ASMTL-AS1
4020,DHRSX-IT1,ENSG00000223571,DHRSX-IT1_1,ENSG00000223571,DHRSX-IT1,ENSG00000223571,DHRSX-IT1
4097,AL732314.6,ENSG00000281849,AL732314.6_2,ENSG00000281849,AL732314.6,ENSG00000281849,ENSG00000281849
4099,LINC00106,ENSG00000236871,LINC00106_2,ENSG00000236871,LINC00106,ENSG00000236871,LINC00106
4101,ASMTL-AS1,ENSG00000236017,ASMTL-AS1_2,ENSG00000236017,ASMTL-AS1,ENSG00000236017,ASMTL-AS1
4103,DHRSX-IT1,ENSG00000223571,DHRSX-IT1_2,ENSG00000223571,DHRSX-IT1,ENSG00000223571,DHRSX-IT1


In [238]:
mask = himorna_rnachrom_mapping_updated["new-ens_gene"].apply(lambda x: "retired" in x)
himorna_rnachrom_mapping_updated[mask]

,gene_name HimoRNA,Ensembl ID HimoRNA,gene_name RNA-Chrom,Ensembl ID RNA-Chrom,gene_name,new-ens_gene,new-gene_name
40,AL109811.2,ENSG00000271895,AL109811.2,ENSG00000271895,AL109811.2,<retired>,<retired>
111,LINC01358,ENSG00000237352,LINC01358,ENSG00000237352,LINC01358,<retired>,<retired>
150,AC093157.2,ENSG00000235795,AC093157.2,ENSG00000235795,AC093157.2,<retired>,<retired>
161,AL365361.1,ENSG00000259834,AL365361.1,ENSG00000259834,AL365361.1,<retired>,<retired>
259,AL356273.3,ENSG00000279838,AL356273.3,ENSG00000279838,AL356273.3,<retired>,<retired>
...,...,...,...,...,...,...,...
3979,AL596244.1,ENSG00000261534,AL596244.1,ENSG00000261534,AL596244.1,<retired>,<retired>
3980,TTLL11-IT1,ENSG00000237548,TTLL11-IT1,ENSG00000237548,TTLL11-IT1,<retired>,<retired>
4054,AL353804.1,ENSG00000228906,AL353804.1,ENSG00000228906,AL353804.1,<retired>,<retired>
4069,AL035425.3,ENSG00000261409,AL035425.3,ENSG00000261409,AL035425.3,<retired>,<retired>


In [240]:
mask = himorna_rnachrom_mapping_updated["new-ens_gene"].apply(lambda x: "ENSG" in x)
himorna_rnachrom_mapping_updated_saved_ensg = himorna_rnachrom_mapping_updated[mask]
himorna_rnachrom_mapping_updated_saved_ensg["new-ens_gene"].nunique()

3974

In [242]:
3974 + 61

4035

In [241]:
assert 3974 + 61 == 4149

AssertionError: 

In [97]:
contacts_test_gencode_v35_to_v47 = pd.merge(left=contacts_test, right=himorna_rnachrom_mapping_updated_saved_ensg[["gene_name RNA-Chrom","new-ens_gene","new-gene_name"]], how="inner", left_on="gene_name", right_on="gene_name RNA-Chrom")
del contacts_test_gencode_v35_to_v47["gene_name"]
contacts_test_gencode_v35_to_v47

,chrom,start,end,strand,gene_name RNA-Chrom,new-ens_gene,new-gene_name
0,chr3,47215955,47216025,+,KIF9-AS1,ENSG00000227398,KIF9-AS1
1,chr5,154224456,154224492,+,AC078785.1,ENSG00000240057,NEPRO-AS1
2,chr3,40265787,40265838,-,EIF1B-AS1,ENSG00000280739,EIF1B-AS1
3,chr3,181253813,181253849,+,SOX2-OT,ENSG00000242808,SOX2-OT
4,chr3,178303536,178303636,-,AC117453.1,ENSG00000223930,ENSG00000223930
...,...,...,...,...,...,...,...
176377,chrX,20207452,20207501,-,FTX,ENSG00000230590,FTX
176378,chr5,87806274,87806341,-,FTX,ENSG00000230590,FTX
176379,chrX,63480664,63480764,-,LINC01278,ENSG00000235437,LINC01278
176380,chrX,115917470,115917570,-,DANT2,ENSG00000235244,DANT2


In [22]:
mask = contacts_test_gencode_v35_to_v47["new-ens_gene"].isna()
contacts_test_gencode_v35_to_v47[mask]["gene_name"].nunique()

71403

In [206]:
dict_of_rnachrom_peaks_by_exp_tissue_version = {}
for project in RNACHROM_INITIAL.glob("experiment*"):
    if project.is_dir:
        print(project.name)
        exp = project.name.split("method_")[1].split(("_organism"))[0]
        print(f"\t{exp}")
        tissue = project.name.split("tissue_")[1]
        print(f"\t{tissue}")
        version = project.name.split("_method")[0].split("id_")[1]
        print(f"\t\t{version}")
        res_concat = pd.concat([pd.read_csv(i, sep="\t") for i in RNACHROM_INITIAL.glob(f"{project.name}/contacts/*.tab")])
        len_1 = len(res_concat)
        res_concat = res_concat.drop_duplicates()
        len_2 = len(res_concat)
        usecols=["dna_chr","dna_start","dna_end","gene_name","gene_strand"]
        res_concat = res_concat[usecols]
        contacts_gencode_v35_to_v47 = pd.merge(left=res_concat, right=himorna_rnachrom_mapping_updated_saved_ensg[["gene_name RNA-Chrom","new-ens_gene","new-gene_name"]], how="inner", left_on="gene_name", right_on="gene_name RNA-Chrom")
        del contacts_gencode_v35_to_v47["gene_name"]
        dict_of_rnachrom_peaks_by_exp_tissue_version[f"{exp}-{tissue}-{version}"] = [contacts_gencode_v35_to_v47, (len_1 - len_2) / len_1 * 100]

experiment_all_to_all_id_1_method_iMARGI_organism_Homo_sapiens_tissue_HFFc6
	iMARGI
	HFFc6
		1
experiment_all_to_all_id_8_method_GRID_organism_Homo_sapiens_tissue_MDA_MB_231__HTB_26_ATCC_
	GRID
	MDA_MB_231__HTB_26_ATCC_
		8
experiment_all_to_all_id_9_method_Red_C_organism_Homo_sapiens_tissue_fibroblasts
	Red_C
	fibroblasts
		9
experiment_all_to_all_id_2_method_iMARGI_organism_Homo_sapiens_tissue_HEK293T
	iMARGI
	HEK293T
		2
experiment_all_to_all_id_3_method_iMARGI_organism_Homo_sapiens_tissue_HUVEC
	iMARGI
	HUVEC
		3
experiment_all_to_all_id_4_method_iMARGI_organism_Homo_sapiens_tissue_HUVEC
	iMARGI
	HUVEC
		4
experiment_all_to_all_id_5_method_iMARGI_organism_Homo_sapiens_tissue_HUVEC
	iMARGI
	HUVEC
		5
experiment_all_to_all_id_7_method_GRID_organism_Homo_sapiens_tissue_MM_1S
	GRID
	MM_1S
		7
experiment_all_to_all_id_10_method_Red_C_organism_Homo_sapiens_tissue_K562
	Red_C
	K562
		10


In [209]:
dict_of_rnachrom_peaks_by_exp_tissue.keys()

dict_keys(['iMARGI-HFFc6-1', 'GRID-MDA_MB_231__HTB_26_ATCC_-8', 'Red_C-fibroblasts-9', 'iMARGI-HEK293T-2', 'iMARGI-HUVEC-3', 'iMARGI-HUVEC-4', 'iMARGI-HUVEC-5', 'GRID-MM_1S-7', 'Red_C-K562-10'])

In [207]:
{i:np.round(dict_of_rnachrom_peaks_by_exp_tissue[i][1], 4) for i in dict_of_rnachrom_peaks_by_exp_tissue}

{'iMARGI-HFFc6-1': 1.5175,
 'GRID-MDA_MB_231__HTB_26_ATCC_-8': 0.005,
 'Red_C-fibroblasts-9': 12.5832,
 'iMARGI-HEK293T-2': 2.8266,
 'iMARGI-HUVEC-3': 26.5105,
 'iMARGI-HUVEC-4': 14.6943,
 'iMARGI-HUVEC-5': 6.3554,
 'GRID-MM_1S-7': 0.0004,
 'Red_C-K562-10': 0.4646}

In [228]:
dict_of_rnachrom_peaks_by_exp_tissue_version["iMARGI-HFFc6-1"][0]

,dna_chr,dna_start,dna_end,gene_strand,gene_name RNA-Chrom,new-ens_gene,new-gene_name
0,chr3,47215955,47216025,+,KIF9-AS1,ENSG00000227398,KIF9-AS1
1,chr5,154224456,154224492,+,AC078785.1,ENSG00000240057,NEPRO-AS1
2,chr3,40265787,40265838,-,EIF1B-AS1,ENSG00000280739,EIF1B-AS1
3,chr3,181253813,181253849,+,SOX2-OT,ENSG00000242808,SOX2-OT
4,chr3,178303536,178303636,-,AC117453.1,ENSG00000223930,ENSG00000223930
...,...,...,...,...,...,...,...
173574,chrX,20207452,20207501,-,FTX,ENSG00000230590,FTX
173575,chr5,87806274,87806341,-,FTX,ENSG00000230590,FTX
173576,chrX,63480664,63480764,-,LINC01278,ENSG00000235437,LINC01278
173577,chrX,115917470,115917570,-,DANT2,ENSG00000235244,DANT2


In [229]:
dict_of_rnachrom_peaks_by_exp_tissue_version["iMARGI-HFFc6-1"][0]["new-ens_gene"].nunique()

3087

In [230]:
for exp_tissue_version, dd in dict_of_rnachrom_peaks_by_exp_tissue_version.items():
    print(exp_tissue_version)
    break

iMARGI-HFFc6-1


In [232]:
def indexing_rnachrom_lncrna_by_bed(exp_tissue_version, lnc_ens, subdf):
    to_index = RNACHROM_INDEX_BED_V2 / exp_tissue_version
    to_index.mkdir(exist_ok=True, parents=True)
    to_save = to_index / f"{exp_tissue_version}-{lnc_ens}-sort.bed"
    if to_save.exists():
        print(f"Skipped: {exp_tissue_version} {lnc_ens}")
        return None
    subdf["name"] = "."
    subdf["score"] = "."
    subdf = subdf[["dna_chr", "dna_start", "dna_end", "name", "score", "gene_strand"]]
    subdf_bed = pybedtools.BedTool.from_dataframe(subdf)
    subdf_bed_sortbed = subdf_bed.sort()
    subdf_bed_sortbed.moveto(to_save)

In [233]:
with Parallel(n_jobs=-1) as pool:
    chunks = pool(
        delayed(indexing_rnachrom_lncrna_by_bed)(exp_tissue_version=exp_tissue_version, lnc_ens=lnc_ens[0], subdf=subdf)
        for exp_tissue_version, dd in dict_of_rnachrom_peaks_by_exp_tissue_version.items()
        for lnc_ens, subdf in dd[0].groupby(by=["new-ens_gene"])
    )

In [225]:
[pd.read_csv(i) for i in (RNACHROM_INDEX_BED / "GRID-MDA_MB_231__HTB_26_ATCC_").glob("GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000093100-sort.bed")]

[          chr1\t8045642\t8045665\t.\t.\t-
 0         chr1\t9584333\t9584355\t.\t.\t-
 1         chr1\t9720817\t9720840\t.\t.\t-
 2       chr1\t10230041\t10230063\t.\t.\t-
 3       chr1\t17878112\t17878134\t.\t.\t-
 4       chr1\t17878112\t17878134\t.\t.\t-
 ...                                   ...
 1369  chrX\t135004207\t135004230\t.\t.\t-
 1370  chrX\t135179552\t135179574\t.\t.\t-
 1371  chrX\t136145565\t136145587\t.\t.\t-
 1372  chrX\t146357759\t146357781\t.\t.\t-
 1373  chrX\t152810062\t152810085\t.\t.\t-
 
 [1374 rows x 1 columns]]

## Аннотация (вроде бы излишняя)

In [10]:
gtf_file_GENCODE35 = DATA_INITIAL / "gencode/gencode.v35.basic.annotation.gtf.gz"
anno_GENCODE35 = pd.read_csv(gtf_file_GENCODE35, sep="\t", header=None, skiprows=5)
anno_GENCODE35.head(2)

,0,1,2,3,4,5,6,7,8
0,chr1,HAVANA,gene,11869,14409,.,+,.,"gene_id ""ENSG00000223972.5""; gene_type ""transc..."
1,chr1,HAVANA,transcript,11869,14409,.,+,.,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."


In [11]:
anno_GENCODE35.shape

(1789637, 9)

In [12]:
anno_GENCODE35["ens_gene"] = anno_GENCODE35[8].apply(lambda x: x.split(" ")[1].replace('"', '').replace(';', '').split(".")[0])

In [13]:
anno_GENCODE35["gene_name"] = anno_GENCODE35[8].apply(lambda x: x.split("gene_name ")[1].split(";")[0].replace('"', ''))

In [14]:
ens_and_name_GENCODE35 = anno_GENCODE35[["ens_gene", "gene_name"]].drop_duplicates()
#ens_and_name_GENCODE35.set_index("gene_name", inplace=True)
ens_and_name_GENCODE35.head(2)

,ens_gene,gene_name
0,ENSG00000223972,DDX11L1
12,ENSG00000227232,WASH7P


In [109]:
ens_and_name_GENCODE35.shape

(60612, 2)

In [15]:
len([i for i in RNACHROM_INITIAL.glob("*/contacts/*.tab")])

363

In [16]:
pd.read_csv([i for i in RNACHROM_INITIAL.glob("*/contacts/*.tab")][0], sep="\t")

,rna_chr,rna_start,rna_end,dna_chr,dna_start,dna_end,sample,gene_name,gene_type,gene_strand,source,expected_value,norm_value
0,chr3,47595901,47595999,chr3,47595961,47596017,SRR8206680,SMARCC1,protein_coding,-,gencode,0.03999,1.007
1,chr3,194463805,194463903,chr7,139874781,139874838,SRR8206680,ATP13A3,protein_coding,-,gencode,0.06362,0.964
2,chr3,47851633,47851731,chr3,47851695,47851752,SRR8206680,MAP4,protein_coding,-,gencode,0.02933,1.027
3,chr3,72421268,72421366,chr3,72631180,72631234,SRR8206680,RYBP,protein_coding,-,gencode,0.02967,1.026
4,chr3,11315978,11316029,chr1,243368102,243368155,SRR8206680,ATG7,protein_coding,+,gencode,0.02830,1.029
...,...,...,...,...,...,...,...,...,...,...,...,...,...
242889,chr3,184819152,184819181,chr3,135445357,135445455,SRR8206680,VPS8,protein_coding,+,gencode,0.00143,1.084
242890,chr3,37045936,37046034,chr3,37045871,37045971,SRR8206680,MLH1,protein_coding,+,gencode,0.10700,0.895
242891,chr3,86005261,86005345,chr3,86005376,86005437,SRR8206680,CADM2-AS1,lncRNA,-,gencode,0.00102,1.085
242892,chr3,53868490,53868588,chr16,59640738,59640780,SRR8206680,ACTR8,protein_coding,-,gencode,0.02338,1.038


In [17]:
test1_chr3 = pd.read_csv([i for i in RNACHROM_INITIAL.glob("experiment_all_to_all_id_1_method_iMARGI_organism_Homo_sapiens_tissue_HFFc6/contacts/*.tab")][0], sep="\t")
test1_chr3.shape

(242894, 13)

In [18]:
ens_and_name_GENCODE35[ens_and_name_GENCODE35["gene_name"] == "SMARCC1"]

,ens_gene,gene_name
332370,ENSG00000173473,SMARCC1


In [115]:
merge = pd.merge(left=ens_and_name_GENCODE35, right=test1_chr3, left_on="gene_name", right_on="gene_name", how="inner")

In [118]:
merge

,ens_gene,gene_name,rna_chr,rna_start,rna_end,dna_chr,dna_start,dna_end,sample,gene_type,gene_strand,source,expected_value,norm_value
0,ENSG00000223587,LINC01986,chr3,13571,13669,chr3,13485,13585,SRR8206680,lncRNA,+,gencode,0.005670,1.075
1,ENSG00000223587,LINC01986,chr3,12701,12799,chr3,12670,12770,SRR8206680,lncRNA,+,gencode,0.004240,1.078
2,ENSG00000223587,LINC01986,chr3,22374,22472,chr3,22373,22473,SRR8206680,lncRNA,+,gencode,0.038730,1.009
3,ENSG00000223587,LINC01986,chr3,22156,22254,chr3,22008,22108,SRR8206680,lncRNA,+,gencode,0.038730,1.009
4,ENSG00000223587,LINC01986,chr3,19365,19463,chr3,19247,19347,SRR8206680,lncRNA,+,gencode,0.032600,1.021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224358,ENSG00000232783,FRG2FP,chr3,198111098,198111196,chr3,198111022,198111122,SRR8206680,transcribed_unprocessed_pseudogene,+,gencode,0.013090,1.059
224359,ENSG00000232783,FRG2FP,chr3,198111072,198111170,chr3,198111022,198111122,SRR8206680,transcribed_unprocessed_pseudogene,+,gencode,0.013090,1.059
224360,ENSG00000232783,FRG2FP,chr3,198111098,198111196,chr3,198111022,198111090,SRR8206680,transcribed_unprocessed_pseudogene,+,gencode,0.013090,1.059
224361,ENSG00000236246,AC073135.1,chr3,198121773,198121871,chr9,138178354,138178454,SRR8206680,unprocessed_pseudogene,-,gencode,0.012650,1.060


## Продолжение